# Staff Planning for Student Events

We want to manage the staff schedule of a BDE event using Python.
For an event, there are different staff positions (e.g. bar, sound, entrance…) and a certain number of staff members.
Staff must rotate at each time slot.

We also want to track each person's staff hours to equalize them.

The user chooses the timing of their slots, and must also be able to choose how many people they want for each position.

**Inputs:**
- **Event time range** (from 9pm to 5am)
- **Rotation timing** (change every 1h / 2h)
- **Name and gender of each staff member**
- **Their constraints** (e.g. doesn't like vomit, the bar…)
- **Number of people required for each position** (default values will likely be set for some positions)

**Outputs:**
- A general Excel file with the full schedule slot by slot
- An individual Excel schedule
- An individual image schedule (phone wallpaper size)

## Phase 2: Additional Constraints
- wants to work with someone
- doesn't want to work with someone
- only available at certain time slots

# Import and Setup

## Cell to run when working locally

In [159]:
import pandas as pd
from datetime import datetime, timedelta
import random

## Class Definitions

In [160]:
##### Staff Member #####
class Staff:
  def __init__(self, last_name, first_name, gender):
        self.last_name = last_name
        self.first_name = first_name
        self.gender = gender
        self.constraints = []
        self.staff_hours = 0

  def __str__(self):
        return self.last_name + " " + self.first_name

##### Role #####
class Role:
  def __init__(self, name, max_staff):
        self.name = name
        self.max_staff = max_staff

  def __str__(self):
        return self.name

##### Time Slot #####
class TimeSlot:
  def __init__(self, start_time, end_time):   # date = datetime(year, month, day, hour)
        self.start_time = start_time
        self.end_time = end_time

  def __str__(self):
        return f"{self.start_time.strftime("%Hh%M")} - {self.end_time.strftime("%Hh%M")}"

  def duration(self):
        return self.end_time - self.start_time


##### Assignment #####
class Assignment:
    def __init__(self, staff, time_slot, role):
        self.staff = staff
        self.time_slot = time_slot
        self.role = role

    def __str__(self):
        return f"{self.staff} --> {self.role} --> {self.time_slot}"


## Tests

In [161]:
# Staff members
s1 = Staff("Martin", "Jean", "M")
s2 = Staff("Dupont", "Michelle", "F")
s3 = Staff("Dupont", "Paul", "M")

# Roles
p1 = Role("Bar", 2)
p2 = Role("Entrance", 1)

# Time slot
start = datetime(2026, 3, 15, 20)
end = datetime(2026, 3, 16, 4)

In [162]:
print(TimeSlot(start, end))
print(TimeSlot(start, end).duration())

20h00 - 04h00
8:00:00


In [163]:
a1 = Assignment(s1, TimeSlot(start, end), p1)
print(a1)

Martin Jean --> Bar --> 20h00 - 04h00


# Main Functions
## Automatic Slot Generation

In [164]:
def generate_slots(start_time, end_time, interval):
  slots = []
  while start_time < end_time:
    slots.append(TimeSlot(start_time, start_time + timedelta(hours=interval)))
    start_time = start_time + timedelta(hours=interval)

  return slots

for s in generate_slots(start, end, 1):
  print(s)

20h00 - 21h00
21h00 - 22h00
22h00 - 23h00
23h00 - 00h00
00h00 - 01h00
01h00 - 02h00
02h00 - 03h00
03h00 - 04h00


## Input: Staff, Positions, and Event Info



In [165]:
def input_roles():
    roles = []
    while True:
      p = Role(input("role: "), int(input("max number of staff: ")))
      roles.append(p)

      again = input("another? (Y/N) ")
      if again == "N":
        break

    return roles

In [166]:
def input_staff(nb_staff):
  staff_list = []
  for i in range(nb_staff):
    s = Staff(input("last name: "), input("first name: "), input("gender: (M/F) "))
    staff_list.append(s)

  while True: 
    encore = input("encore? (O/N) ")
    if encore == "N":
      break
    s = Staff(input("last name: "), input("first name: "), input("gender: (M/F) "))
    staff_list.append(s)

  return staff_list

In [167]:
def event():
  start_time = datetime.strptime(input("Start date and time: (dd/mm/YYYY  HHhMM)"), "%d/%m/%Y %Hh%M")
  end_time = datetime.strptime(input("End date and time: (dd/mm/YYYY  HHhMM)"), "%d/%m/%Y %Hh%M")

  interval = int(input("Slot duration (hours): "))

  return generate_slots(start_time, end_time, interval)

## Test Data

In [168]:
staff_members = [
    Staff("Martin", "Jean", "M"),
    Staff("Dupont", "Michelle", "F"),
    Staff("Bernard", "Paul", "M"),
    Staff("Petit", "Sophie", "F"),
    Staff("Durand", "Lucas", "M"),
    Staff("Moreau", "Emma", "F"),
]

roles = [
    Role("Bar", 2),
    Role("Entrance", 1),
    Role("Sound", 1),
]

slots = generate_slots(datetime(2026, 4, 3, 21), datetime(2026, 4, 4, 5), 1)

# Assignment Algorithm

1. Select a time slot (schedule + position)
2. Select the person with the fewest staff hours (sort)
3. Check if the person is available, otherwise select another
4. Assign this person to the slot
5. Repeat for each position then for subsequent hours

In [169]:
def assign(staff_members, roles, slots):
    assignments = []

    for slot in slots:
        role_list = roles.copy()
        random.shuffle(role_list)
        for p in role_list:
          for place in range(p.max_staff):
              # 1. Sort staff by staff_hours
              sorted_staff = sorted(staff_members, key=lambda s: s.staff_hours)

              # 2. Find the first available and create the assignment
              for staff in sorted_staff:
                if not any(a.staff == staff and a.time_slot == slot for a in assignments):
                  assignments.append(Assignment(staff, slot, p))
                  sorted_staff.remove(staff)

              # 3. Update staff_hours
                  staff.staff_hours += slot.duration().seconds // 3600
                  break
        # If all positions are filled, assign "Free" to remaining staff
        for staff in sorted_staff:
            if not any(a.staff == staff and a.time_slot == slot for a in assignments):
                assignments.append(Assignment(staff, slot, "Free"))

    
    return assignments

### Test


In [170]:
for a in assign(staff_members, roles, slots):
    print(a)

assignments = assign(staff_members, roles, slots)

for s in staff_members:
    print(s, s.staff_hours, "h")

Martin Jean --> Bar --> 21h00 - 22h00
Dupont Michelle --> Bar --> 21h00 - 22h00
Bernard Paul --> Sound --> 21h00 - 22h00
Petit Sophie --> Entrance --> 21h00 - 22h00
Durand Lucas --> Free --> 21h00 - 22h00
Moreau Emma --> Free --> 21h00 - 22h00
Durand Lucas --> Entrance --> 22h00 - 23h00
Moreau Emma --> Bar --> 22h00 - 23h00
Martin Jean --> Bar --> 22h00 - 23h00
Dupont Michelle --> Sound --> 22h00 - 23h00
Bernard Paul --> Free --> 22h00 - 23h00
Petit Sophie --> Free --> 22h00 - 23h00
Bernard Paul --> Sound --> 23h00 - 00h00
Petit Sophie --> Entrance --> 23h00 - 00h00
Durand Lucas --> Bar --> 23h00 - 00h00
Moreau Emma --> Bar --> 23h00 - 00h00
Martin Jean --> Free --> 23h00 - 00h00
Dupont Michelle --> Free --> 23h00 - 00h00
Martin Jean --> Entrance --> 00h00 - 01h00
Dupont Michelle --> Bar --> 00h00 - 01h00
Bernard Paul --> Bar --> 00h00 - 01h00
Petit Sophie --> Sound --> 00h00 - 01h00
Durand Lucas --> Free --> 00h00 - 01h00
Moreau Emma --> Free --> 00h00 - 01h00
Durand Lucas --> Entranc

## Excel Generation

In [171]:
def generate_general_schedule(staff_members, assignments, slots):
    schedule = pd.DataFrame(
        columns=["Staff Hours"] + [str(s) for s in slots],
        index=[str(m) for m in staff_members]
    )

    for a in assignments:
        schedule.loc[str(a.staff), str(a.time_slot)] = str(a.role)

    for m in staff_members:
        schedule.loc[str(m), "Staff Hours"] = f"{m.staff_hours} h"
    
    schedule.index.name = "Name"
    schedule.sort_values("Name", inplace=True)
    schedule.to_excel("schedule.xlsx")

    schedule

generate_general_schedule(staff_members, assignments, slots)

In [172]:
def generate_individual_schedule(staff_members, assignments, slots):
    workbook = xlsxwriter.Workbook('individual_schedule.xlsx')

    for s in staff_members:
        row=0
        col=0
        
        worksheet = workbook.add_worksheet(str(s))
        worksheet.write(row, col, "Time")
        worksheet.write(row, col+1, "Role")
        row+=1
        for slot in slots:
            for a in assignments:
                if a.staff == s and a.time_slot == slot:
                    worksheet.write(row, col, str(slot))
                    worksheet.write(row, col+1, str(a.role))
                    row+=1


    workbook.close()

generate_individual_schedule(staff_members, assignments, slots)

## Main Program

In [173]:
def run_staff_planner():
    positions = input_roles()
    nb_staff = sum(p.max_staff for p in roles)
    print(nb_staff)
    staff_members = input_staff(nb_staff)
    slots = event()
    assignments = assign(staff_members, roles, slots)
    generate_general_schedule(staff_members, assignments, slots)
    generate_individual_schedule(staff_members, assignments, slots)

In [174]:
run_staff_planner()

4


ValueError: time data '24/01/2027' does not match format '%d/%m/%Y %Hh%M'